# PHẦN 1: TẢI DỮ LIỆU VÀ KIỂM TRA CẤU TRÚC BAN ĐẦU

Cần có file `cardio_train.csv` nằm cùng thư mục với notebook này.
(Nếu chạy trên Google Colab: bấm biểu tượng thư mục bên trái, rồi upload file lên trước khi chạy.)

#### 1.1: Đọc dữ liệu

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Đọc file (file cardio dùng dấu ; để ngăn cách các cột)
df_raw = pd.read_csv("cardio_train.csv", sep=";")

# Nếu file dùng dấu phẩy thì đọc lại
if df_raw.shape[1] == 1:
    df_raw = pd.read_csv("cardio_train.csv")

# Đưa tên cột về chữ thường, bỏ khoảng trắng thừa
df_raw.columns = df_raw.columns.str.strip().str.lower()

# Lưu lại vài con số ban đầu để so sánh ở bước 2.5
so_hang_ban_dau = len(df_raw)
so_cot_ban_dau = df_raw.shape[1]
so_thieu_ban_dau = df_raw.isnull().sum().sum()

# Đổi tuổi từ ngày sang năm (làm tròn thành số nguyên)
df_raw["age_years"] = (df_raw["age"] / 365.25).round(0).astype(int)

df_raw[["age", "age_years"]].head()

#### 1.2: Tổng quan dữ liệu

In [ ]:
print("Kích thước:", df_raw.shape[0], "hàng,", df_raw.shape[1], "cột")
print("Các cột:", list(df_raw.columns))

In [ ]:
# 5 hàng đầu
df_raw.head()

In [ ]:
# 5 hàng cuối
df_raw.tail()

In [ ]:
# Kiểu dữ liệu và số giá trị không rỗng của từng cột
df_raw.info()

In [ ]:
# Thống kê cơ bản (trung bình, nhỏ nhất, lớn nhất, ...)
df_raw.describe().T

#### 1.3: Biểu đồ tròn cho các cột phân loại

In [ ]:
cot_phan_loai = ["gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]

for cot in cot_phan_loai:
    dem = df_raw[cot].value_counts().sort_index()

    plt.figure(figsize=(5, 5))
    plt.pie(dem, labels=dem.index, autopct="%1.1f%%", startangle=90)
    plt.title("Phân bố của " + cot)
    plt.show()

#### 1.4: Biểu đồ tần suất của huyết áp (ap_hi và ap_lo)

Chia giá trị thành các khoảng rộng 10 (ví dụ 120 đến 130), chỉ vẽ các khoảng có từ 2000 bản ghi trở lên.

In [ ]:
for cot in ["ap_hi", "ap_lo"]:
    # Ví dụ: 125 -> 120, 138 -> 130
    khoang = df_raw[cot] // 10 * 10

    # Đếm số bản ghi trong mỗi khoảng, chỉ giữ khoảng có từ 2000 bản ghi
    tan_suat = khoang.value_counts().sort_index()
    tan_suat = tan_suat[tan_suat >= 2000]

    # Tạo nhãn dạng "120-130"
    nhan = []
    for k in tan_suat.index:
        nhan.append(str(k) + "-" + str(k + 10))

    plt.figure(figsize=(10, 5))
    plt.bar(nhan, tan_suat.values, edgecolor="black")
    plt.title("Tần suất của " + cot)
    plt.xlabel("Khoảng huyết áp (mmHg)")
    plt.ylabel("Số bản ghi")
    plt.xticks(rotation=45)
    plt.show()

    print(cot, "- nhỏ nhất:", df_raw[cot].min(), "- lớn nhất:", df_raw[cot].max())

#### 1.5: Biểu đồ phân bố của các cột số còn lại

In [ ]:
for cot in ["height", "weight", "age_years"]:
    plt.figure(figsize=(6, 4))
    plt.hist(df_raw[cot], bins=30, edgecolor="black")
    plt.title("Phân bố của " + cot)
    plt.xlabel(cot)
    plt.ylabel("Số bản ghi")
    plt.show()

# PHẦN 2: LÀM SẠCH VÀ TIỀN XỬ LÝ DỮ LIỆU

#### 2.1: Kiểm tra và xử lý bản ghi trùng lặp

In [ ]:
# Trùng cả cột id
so_trung_ca_id = df_raw.duplicated().sum()

# Trùng tất cả các cột trừ id (cùng một bệnh nhân nhưng khác id)
cot_khong_id = list(df_raw.columns)
cot_khong_id.remove("id")
so_trung = df_raw.duplicated(subset=cot_khong_id).sum()

print("Số hàng trùng (tính cả id):", so_trung_ca_id)
print("Số hàng trùng (không tính id):", so_trung)

# Giữ lại hàng xuất hiện đầu tiên, xóa các hàng trùng
df_clean = df_raw.drop_duplicates(subset=cot_khong_id).copy()

print("Số hàng trước:", len(df_raw))
print("Số hàng sau  :", len(df_clean))
print("Số hàng đã xóa:", len(df_raw) - len(df_clean))

#### 2.2: Kiểm tra và xử lý giá trị thiếu

Bỏ các cột rỗng hoàn toàn, sau đó bỏ các hàng có giá trị thiếu.

In [ ]:
print("Số giá trị thiếu của từng cột:")
print(df_clean.isnull().sum())

# Bỏ cột nào rỗng hoàn toàn
df_clean = df_clean.dropna(axis=1, how="all")

# Bỏ hàng nào có ít nhất 1 giá trị thiếu
so_hang_truoc = len(df_clean)
df_clean = df_clean.dropna().copy()

print("Số hàng đã xóa:", so_hang_truoc - len(df_clean))
print("Số hàng còn lại:", len(df_clean))
print("Số giá trị thiếu còn lại:", df_clean.isnull().sum().sum())

#### 2.3: Chuẩn hóa kiểu dữ liệu và định dạng

In [ ]:
# Ép các cột về dạng số (giá trị không chuyển được sẽ thành NaN)
cot_so = ["id", "age", "gender", "height", "weight", "ap_hi", "ap_lo",
          "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]

for cot in cot_so:
    df_clean[cot] = pd.to_numeric(df_clean[cot], errors="coerce")

# Nếu bước trên làm xuất hiện giá trị thiếu thì điền lại:
# cột phân loại điền giá trị xuất hiện nhiều nhất, cột đo điền trung vị
for cot in cot_phan_loai:
    df_clean[cot] = df_clean[cot].fillna(df_clean[cot].mode()[0])

for cot in ["age", "height", "weight", "ap_hi", "ap_lo"]:
    df_clean[cot] = df_clean[cot].fillna(df_clean[cot].median())

In [ ]:
# Kiểm tra các cột phân loại có giá trị nào lạ không
ma_hop_le = {
    "gender": [1, 2],
    "cholesterol": [1, 2, 3],
    "gluc": [1, 2, 3],
    "smoke": [0, 1],
    "alco": [0, 1],
    "active": [0, 1],
    "cardio": [0, 1],
}

for cot in ma_hop_le:
    quan_sat = sorted(df_clean[cot].unique().tolist())

    gia_tri_la = []
    for gia_tri in quan_sat:
        if gia_tri not in ma_hop_le[cot]:
            gia_tri_la.append(gia_tri)

    print(cot, "- giá trị quan sát:", quan_sat, "- giá trị lạ:", gia_tri_la)

In [ ]:
# Kiểu dữ liệu của các cột sau khi chuẩn hóa
df_clean.dtypes

#### 2.4: Nhận diện và xử lý ngoại lệ ban đầu

Chỉ kiểm tra các cột đo liên tục (không kiểm tra các cột phân loại và cột 0/1).

In [ ]:
cot_lien_tuc = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

fig, axes = plt.subplots(1, len(cot_lien_tuc), figsize=(17, 4))

for i in range(len(cot_lien_tuc)):
    cot = cot_lien_tuc[i]
    axes[i].boxplot(df_clean[cot])
    axes[i].set_title(cot)

plt.suptitle("Boxplot các biến đo trước khi lọc giá trị bất hợp lý")
plt.tight_layout()
plt.show()

In [ ]:
# Đếm ngoại lệ theo quy tắc 1.5 * IQR
ket_qua = []

for cot in cot_lien_tuc:
    q1 = df_clean[cot].quantile(0.25)
    q3 = df_clean[cot].quantile(0.75)
    iqr = q3 - q1
    can_duoi = q1 - 1.5 * iqr
    can_tren = q3 + 1.5 * iqr
    so_ngoai_le = ((df_clean[cot] < can_duoi) | (df_clean[cot] > can_tren)).sum()

    ket_qua.append({
        "Cột": cot,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Cận dưới": can_duoi,
        "Cận trên": can_tren,
        "Số ngoại lệ IQR": so_ngoai_le,
    })

bang_iqr = pd.DataFrame(ket_qua).set_index("Cột")
bang_iqr.round(2)

In [ ]:
# Chỉ giữ lại các hàng có giá trị nằm trong miền hợp lý
# (dấu & nghĩa là "và", tất cả điều kiện phải cùng đúng)
hop_le = (
    df_clean["age_years"].between(18, 100)
    & df_clean["height"].between(120, 220)
    & df_clean["weight"].between(30, 250)
    & df_clean["ap_hi"].between(60, 250)
    & df_clean["ap_lo"].between(40, 150)
    & (df_clean["ap_hi"] > df_clean["ap_lo"])   # huyết áp tâm thu phải lớn hơn tâm trương
)

so_hang_truoc = len(df_clean)
so_hang_bat_hop_ly = (~hop_le).sum()   # dấu ~ nghĩa là "không"
df_clean = df_clean[hop_le].copy()

print("Số hàng bất hợp lý đã xóa:", so_hang_bat_hop_ly)
print("Số hàng còn lại:", len(df_clean), "/", so_hang_truoc)

#### 2.5: Bảng so sánh trước và sau làm sạch

In [ ]:
bang_so_sanh = pd.DataFrame({
    "Chỉ số": [
        "Tổng số hàng",
        "Tổng số cột",
        "Giá trị thiếu",
        "Hàng trùng (không tính id)",
        "Ngoại lệ / giá trị không hợp lý",
    ],
    "Trước làm sạch": [
        so_hang_ban_dau,
        so_cot_ban_dau,
        so_thieu_ban_dau,
        so_trung,
        bang_iqr["Số ngoại lệ IQR"].sum(),
    ],
    "Sau làm sạch": [
        len(df_clean),
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated(subset=cot_khong_id).sum(),
        "Đã lọc miền bất hợp lý",
    ],
    "Phương pháp / Lý do": [
        "Bỏ " + str(so_trung) + " hàng trùng và " + str(so_hang_bat_hop_ly) + " hàng bất hợp lý",
        "Thêm cột age_years (tuổi theo năm)",
        "Bỏ cột rỗng và hàng thiếu giá trị (nếu có)",
        "drop_duplicates(), giữ hàng đầu tiên",
        "IQR để nhận diện, chỉ xóa hàng ngoài miền hợp lý hoặc ap_hi <= ap_lo",
    ],
})

bang_so_sanh

#### 2.6: Xuất file dữ liệu đã làm sạch

File `cleaned_cardio.csv` sẽ được lưu cùng thư mục với notebook.

In [ ]:
df_clean.to_csv("cleaned_cardio.csv", index=False)

print("Đã lưu file cleaned_cardio.csv")
print("Kích thước:", df_clean.shape[0], "hàng,", df_clean.shape[1], "cột")

# Nếu chạy trên Colab và muốn tải file về máy, bỏ dấu # ở 2 dòng dưới:
# from google.colab import files
# files.download("cleaned_cardio.csv")

In [ ]:
# Xem thử vài hàng đầu của dữ liệu đã làm sạch
df_clean.head()